# 🧩 Class 8: Putting It All Together: an AI Agent for SAP
### From pieces to a real solution · Day 4 · Afternoon · 2 hours

**Same story.** Neo can write (Class 3–4), read our documents (Class 5), use tools (Class 6) and plug into systems safely (Class 7). Today we assemble **one assistant**: the **SAP Operations Assistant**.

**Today's question:** *How do LLMs, prompts, knowledge and agents combine into a real business solution, and where do we start?*

```
 User question
   → Prompt & rules                      (Classes 3–4)
   → Agent decides the steps             (Class 6)
   → Company knowledge via RAG           (Class 5)
     + System tools via MCP              (Class 7)
   → LLM writes the answer               (Class 2)
   → Checked, structured response back to the user
```

| Part | Topic | Cells |
|---|---|---|
| 1 | Start simple: choose the simplest approach that works | S, 1–2 |
| 2 | Assemble the parts: knowledge + tools + rules + agent | 3–6 |
| 3 | Business concerns: checks, cost, speed, privacy, oversight | 7–9 |
| 4 | Capstone: your assistant | 10–12 |

> 🔒 All data is **mock** (*Nova Industries*). Everything runs on this laptop with Ollama.

## ⚙️ ▶ CELL S: Setup
Checks Ollama (chat + embedding model) and starts the SAP Orders MCP server from Class 7. Run once.

In [1]:
# ▶ CELL S · SETUP: check Ollama, then start the SAP Orders MCP server (from Class 7). Run once.
import sys, os, re, json, glob
sys.path.insert(0, "../../shared")            # where bootcamp_llm.py and mcp_client.py live
from bootcamp_llm import *
from mcp_client import *

DOCS_DIR = "../data/docs"                     # the mock company documents (Nova Industries)
SERVER = [sys.executable, "../../Class_7_MCP_Guardrails/mcp_server/sap_orders_server.py"]

check_setup(need_embed=True)
print("🤖 Neo says:", ask_ai("Reply with exactly these words: Connection OK", temperature=0, max_tokens=10))
mcp = McpClient(SERVER + ["--allow-write"])   # the server CAN write; a person-approval gate protects it (CELL 9)
print(f"🔌 MCP server: {mcp.server_info['title']} · {len(mcp.list_tools())} tools")

✅ Ollama 0.40.13 is running at http://localhost:11434
✅ Main model : qwen3:8b
✅ Small model: llama3.2:1b
✅ Embed model: nomic-embed-text
⚙️  Config: C:\Users\rajuk\OneDrive\Desktop\Day_4\Day_4\bootcamp_config.json


🤖 Neo says: Connection OK
🔌 MCP server: Nova SAP Orders (mock) · 6 tools


---
# 1️⃣ Start simple
**In real life:** you do not call a consultant to add two numbers. Pick the **simplest approach that works**.

| Need | Use |
|---|---|
| Summarise, draft, classify text | **A good prompt** |
| Answer from company documents | **RAG** |
| Multi-step tasks using systems | **An agent** |

### ▶ CELL 1: Prompt only

In [2]:
# ▶ CELL 1 · PROMPT ONLY: summarise, classify, draft. No documents, no tools.
EMAIL = "Hello, the truck for PO 4500012346 was held up by heavy rain near Pune. We can deliver on Monday instead of Friday. Regards, Rahul"
print("SUMMARISE:", ask_ai(f"Summarise in one sentence:\n{EMAIL}", temperature=0))
print("CLASSIFY :", ask_ai(f"Classify this email as one word: delay, invoice, complaint or thanks.\n{EMAIL}", temperature=0))
print("DRAFT    :", ask_ai("Draft a two-line polite reply asking Rahul to confirm the new delivery date in writing.", temperature=0))

SUMMARISE: The truck for PO 4500012346 was delayed due to heavy rain near Pune, and delivery is now scheduled for Monday instead of Friday.


CLASSIFY : delay


DRAFT    : Could you kindly confirm the new delivery date in writing at your earliest convenience? We look forward to your response.


> ✍️ Three useful jobs, **no documents and no tools**. A good prompt is the cheapest and fastest option. Start here.

### ▶ CELL 2: Can the model choose the approach?

In [3]:
# ▶ CELL 2 · THE SIMPLEST APPROACH THAT WORKS: let the model choose, then check it in code
ROUTER_RULES = """You choose the simplest approach that works. Reply with ONE JSON object: {"approach": "prompt" | "rag" | "agent", "why": "<one short reason>"}
prompt = summarise, draft or classify text that is already given
rag    = answer from company documents (policies, process guides, handbooks)
agent  = multi-step work that needs live system data and several tools"""

NEEDS = [  # (the need, the approach we expect)
    ("Summarise this supplier email in one line.",                                        "prompt"),
    ("What does our policy say about changing vendor bank details?",                      "rag"),
    ("What does IDoc status 51 mean?",                                                    "rag"),
    ("Why is PO 4500012348 blocked, who must approve it and who is the vendor contact?",  "agent"),
    ("Which orders for plant 1000 are blocked?",                                          "agent"),
]
score = 0
for need, expected in NEEDS:
    pick = chat_json([{"role": "system", "content": ROUTER_RULES}, {"role": "user", "content": need}])
    ok = pick.get("approach") == expected
    score += ok
    print(f"{'✅' if ok else '⚠️'} {pick.get('approach'):<7} (expected {expected:<6}) {need}")
print(f"\n{score} / {len(NEEDS)}")

✅ prompt  (expected prompt) Summarise this supplier email in one line.


✅ rag     (expected rag   ) What does our policy say about changing vendor bank details?


✅ rag     (expected rag   ) What does IDoc status 51 mean?


⚠️ rag     (expected agent ) Why is PO 4500012348 blocked, who must approve it and who is the vendor contact?


✅ agent   (expected agent ) Which orders for plant 1000 are blocked?

4 / 5


> 🧭 A ⚠️ is the lesson: **choosing the approach is itself a decision to test.** In a real project a person (or a simple rule) usually makes this choice once, per use case.

---
# 2️⃣ Assemble the parts
Knowledge from Class 5 + tools from Class 7 + rules from Classes 3–4 + the agent from Class 6.

### ▶ CELL 3: Knowledge: RAG, wrapped as a tool

In [4]:
# ▶ CELL 3 · KNOWLEDGE (Class 5): cut the documents, store fingerprints, and wrap the search as a function
def cut(path):
    """Split one document into chunks: one chunk per numbered clause, labelled with its source and section."""
    name = os.path.basename(path).replace(".txt", "")
    heading, chunks = "", []
    for block in open(path, encoding="utf-8").read().split("\n\n"):
        for line in block.strip().splitlines():
            m = re.match(r"^(\d+\.\d+)\s+(.*)", line)
            if m:                                    # a numbered clause, e.g. "3.3 POs above ..."
                chunks.append({"id": f"{name} §{m.group(1)}", "text": f"[{name} · {heading}] {line}"})
            elif re.match(r"^\d+\.\s", line):        # a section heading, e.g. "3. PO APPROVAL LIMITS"
                heading = line
    return chunks

files = sorted(glob.glob(f"{DOCS_DIR}/*.txt"))
chunks = [c for f in files for c in cut(f)]
store = list(zip(chunks, embed([c["text"] for c in chunks])))
print(f"{len(chunks)} chunks from {len(files)} documents stored.")

def search_policy(query, top=3):
    """FIND: the best clauses for a question, with their ids (so answers can cite them)."""
    q = embed(query)
    ranked = sorted(store, key=lambda s: cosine(q, s[1]), reverse=True)[:top]
    return [{"id": c["id"], "text": c["text"], "score": round(cosine(q, v), 2)} for c, v in ranked]

def rag_answer(question):
    """The 4 steps in one go: find the clauses, then answer ONLY from them, with a citation."""
    context = "\n".join(f"[{h['id']}] {h['text']}" for h in search_policy(question))
    rules = "Answer ONLY from the CONTEXT. If the answer is not there, reply exactly: Not in the documents. End with the source in brackets."
    return ask_ai(f"CONTEXT:\n<<<\n{context}\n>>>\n\nQuestion: {question}", system=rules, temperature=0)

for h in search_policy("Who signs off on really big orders?"):
    print(f"  {h['score']}  {h['id']}")

39 chunks from 4 documents stored.


  0.57  Procurement_Policy §2.1
  0.55  Procurement_Policy §3.1
  0.54  Travel_Policy §3.2


> 📖 Same four steps as Class 5. The new part: `search_policy` is just a **function that returns clauses with ids**. Any function can become a tool.

### ▶ CELL 4: Tools: the MCP server and the knowledge tool, on ONE menu

In [5]:
# ▶ CELL 4 · TOOLS (Class 7): MCP tools from the server + our knowledge tool, on ONE menu
class ToolHub:
    """One menu for the agent. MCP tools come from the server; the knowledge tool is our own function. The agent cannot tell the difference."""
    def __init__(self, mcp, local): self.mcp, self.local = mcp, local
    def list_tools(self): return self.mcp.list_tools() + [spec for spec, _ in self.local.values()]
    def call_tool(self, name, args):
        if name in self.local:
            try: return {"ok": True, "data": self.local[name][1](**args)}
            except TypeError as e: return {"ok": False, "data": f"Bad arguments for {name}: {e}"}
        return self.mcp.call_tool(name, args)

KNOWLEDGE_TOOLS = {"search_policy": ({
    "name": "search_policy",
    "description": "Search the company documents (procurement policy, P2P process guide, travel policy, IDoc handbook). Returns the 3 best clauses with their ids. Use it for rules and procedures.",
    "inputSchema": {"type": "object", "properties": {"query": {"type": "string", "description": "What to look up, in plain words"}}, "required": ["query"]},
    "annotations": {"readOnlyHint": True}}, search_policy)}

client = ToolHub(mcp, KNOWLEDGE_TOOLS)
for t in client.list_tools():
    print(f" • {t['name']:<16} ← {'knowledge (RAG, Class 5)' if t['name'] in KNOWLEDGE_TOOLS else 'MCP server (Class 7)'}")

 • list_orders      ← MCP server (Class 7)
 • get_order        ← MCP server (Class 7)
 • get_vendor       ← MCP server (Class 7)
 • check_approval   ← MCP server (Class 7)
 • delayed_summary  ← MCP server (Class 7)
 • create_ticket    ← MCP server (Class 7)
 • search_policy    ← knowledge (RAG, Class 5)


> 🔌 Live SAP data comes **through MCP**. Company documents come **through RAG**. To the agent both are just tools on a menu.

### ▶ CELL 5: Prompt and rules: the assistant's role, rules and format

In [6]:
# ▶ CELL 5 · PROMPT & RULES (Classes 3-4): the assistant's role, rules and output format
ROLE = "You are the SAP Operations Assistant for Nova Industries. You handle multi-step questions about purchase orders, vendors and company policy."
RULES = [   # ← change me: add or edit a rule
    "Use ONLY facts returned by your tools. Never guess a status, a value, a contact, an approver or a policy.",
    "Use search_policy for company rules and cite the clause id in square brackets, for example [Procurement_Policy §6.2].",
    "Use the order, vendor and approval tools for live SAP data.",
    "You may only READ. If someone asks for a change, ask for a ticket and wait for a person to approve it.",
    "If a tool returns an error or the answer is not available, say what is missing. Do not invent it.",
]
FORMAT = "Write the final answer in at most four short sentences. Start with the main finding."      # ← change me

ALLOWED = None            # None = every tool the server offers. Or a set of names: an allow-list.
MAX_STEPS = 6             # the step limit: an agent must never loop forever

def offered():
    """The tools the server offers RIGHT NOW (name -> description, arguments, read-only label). Nobody types this list."""
    return {t["name"]: t for t in client.list_tools() if ALLOWED is None or t["name"] in ALLOWED}

def system_prompt():
    menu = "\n".join(f"- {n}({', '.join(t['inputSchema']['properties'])}): {t['description']}" for n, t in offered().items())
    rules = "\n".join(f"- {r}" for r in RULES)
    return f"""{ROLE}
RULES:
{rules}

TOOLS (the only ones you may use):
{menu}

Reply with ONE JSON object and nothing else.
To use a tool:  {{"thought": "why I need it", "tool": "<tool name>", "args": {{"<argument>": "<value>"}}}}
To finish:      {{"thought": "why I can finish", "answer": "<final answer for the user>"}}
{FORMAT}""".strip()

print(system_prompt())

You are the SAP Operations Assistant for Nova Industries. You handle multi-step questions about purchase orders, vendors and company policy.
RULES:
- Use ONLY facts returned by your tools. Never guess a status, a value, a contact, an approver or a policy.
- Use search_policy for company rules and cite the clause id in square brackets, for example [Procurement_Policy §6.2].
- Use the order, vendor and approval tools for live SAP data.
- You may only READ. If someone asks for a change, ask for a ticket and wait for a person to approve it.
- If a tool returns an error or the answer is not available, say what is missing. Do not invent it.

TOOLS (the only ones you may use):
- list_orders(plant, status, vendor_id): List purchase orders. Optional filters: plant ('1000' or '2000'), status (Open, Delayed, Blocked, Delivered), vendor_id (V100 to V400). Each order has po_number, vendor, plant, material, value_inr, delivery_date, status and block_reason.
- get_order(po_number): Full details of ON

> 📝 The assistant's **identity is data**: a role, a list of rules and a format. Change a rule and the behaviour changes. Yesterday's menu and today's rules are assembled into one system prompt.

### ▶ CELL 6: The agent: one multi-step question, end to end

In [7]:
# ▶ CELL 6 · THE AGENT (Class 6) with KNOWLEDGE + TOOLS: one multi-step question, end to end
def gate(tool, args):
    """GUARDRAIL between 'the AI asked' and 'the program runs it'. First version: read-only tools only."""
    return bool(tool["annotations"].get("readOnlyHint"))

def execute(reply):
    """The PROGRAM's side: check the request, then ask the MCP server to run it."""
    name, args = reply.get("tool"), reply.get("args")
    tools = offered()
    if name not in tools:
        return {"error": f"Unknown tool '{name}'. Allowed tools: {list(tools)}"}
    if not isinstance(args, dict):
        return {"error": 'args must be an object, for example {"po_number": "4500012345"}'}
    if not gate(tools[name], args):
        return {"error": f"Blocked by a guardrail: '{name}' is not allowed to run."}
    result = client.call_tool(name, args)
    return result["data"] if result["ok"] else {"error": result["data"]}

def run_agent(question, max_steps=None, verbose=True):
    max_steps = max_steps or MAX_STEPS
    messages = [{"role": "system", "content": system_prompt()}, {"role": "user", "content": question}]
    used, evidence = [], []
    for step in range(1, max_steps + 1):
        try:
            reply = chat_json(messages)                                  # THINK
        except ValueError as e:                                          # the model broke the JSON format
            if verbose: print(f"⚠️  step {step}: {e}")
            messages.append({"role": "user", "content": "Reply with ONE valid JSON object only."})
            continue
        if not isinstance(reply, dict):
            messages.append({"role": "user", "content": "Reply with ONE JSON object only."})
            continue
        if verbose: print(f"🧠 THINK   (step {step}): {reply.get('thought', '')}")
        if "answer" in reply:
            if verbose: print(f"✅ ANSWER : {reply['answer']}")
            return {"answer": str(reply["answer"]), "tools": used, "steps": step, "stopped": False, "evidence": evidence, "approval_tools": needs_approval(used)}
        observation = execute(reply)                                     # ACT
        used.append(reply.get("tool"))
        evidence.append({"tool": reply.get("tool"), "args": reply.get("args"), "result": observation})
        if verbose:
            print(f"🛠️  ACT     : {reply.get('tool')}({reply.get('args')})")
            print(f"👀 OBSERVE: {json.dumps(observation, ensure_ascii=False)[:300]}")
        messages += [{"role": "assistant", "content": json.dumps(reply, ensure_ascii=False)},          # OBSERVE
                     {"role": "user", "content": "Tool result: " + json.dumps(observation, ensure_ascii=False)}]
    if verbose: print("⛔ Stopped: step limit reached")                   # a guardrail
    return {"answer": None, "tools": used, "steps": max_steps, "stopped": True, "evidence": evidence, "approval_tools": needs_approval(used)}

def needs_approval(tool_names):
    """Which of these tools change data (so a person must approve them)? The server labels every tool."""
    tools = {t["name"]: t for t in client.list_tools()}
    return [n for n in tool_names if n in tools and not tools[n]["annotations"].get("readOnlyHint", False)]

CAPSTONE_Q = "PO 4500012348 is blocked. Why, who must approve it, what does our policy say about price variance, and who is the vendor contact?"
r = run_agent(CAPSTONE_Q)

🧠 THINK   (step 1): I need to check the block reason, approval requirements, policy on price variance, and vendor contact for PO 4500012348.
🛠️  ACT     : get_order({'po_number': '4500012348'})
👀 OBSERVE: {"po_number": "4500012348", "vendor_id": "V300", "vendor_name": "Chennai Components", "plant": "1000", "material": "M-400 Gearbox", "qty": 10, "value_inr": 1200000, "delivery_date": "2026-09-30", "status": "Blocked", "block_reason": "Price variance"}


🧠 THINK   (step 2): I now have the block reason, vendor ID, and value. I need to find the approval requirements and policy on price variance.


🛠️  ACT     : search_policy({'query': 'price variance approval'})
👀 OBSERVE: [{"id": "Procurement_Policy §6.2", "text": "[Procurement_Policy · 6. INVOICE DISPUTES] 6.2 A price variance above 1% between the PO and the invoice blocks the invoice for review.", "score": 0.63}, {"id": "Procurement_Policy §3.3", "text": "[Procurement_Policy · 3. PO APPROVAL LIMITS] 3.3 POs above ₹


🧠 THINK   (step 3): I now have the block reason, approval requirements, and policy details. I need to find the vendor contact.
🛠️  ACT     : get_vendor({'vendor_id': 'V300'})
👀 OBSERVE: {"vendor_id": "V300", "vendor": "Chennai Components", "contact": "Lakshmi Iyer", "email": "lakshmi@chennai.example", "rating": "B"}


🧠 THINK   (step 4): I have all the required information: the block reason, approval requirements, policy on price variance, and vendor contact.
✅ ANSWER : PO 4500012348 is blocked due to price variance. The policy states that price variances above 1% block the invoice for review [Procurement_Policy §6.2]. This PO requires Director approval as it exceeds ₹10,00,000 [Procurement_Policy §3.3]. The vendor contact is Lakshmi Iyer at lakshmi@chennai.example.


> 🧩 Four steps, **three kinds of source**: the order (MCP), the approver (MCP), the policy clause (RAG) and the vendor (MCP). Nobody wrote this sequence; the agent chose it. Expected facts: **blocked for price variance**, **Director** (₹12,00,000), **Procurement_Policy §6.2**, contact **Lakshmi Iyer**.

---
# 3️⃣ Business concerns
A demo is not a solution. Five questions every sponsor will ask: **accuracy · cost · speed · data privacy · human oversight.**

### ▶ CELL 7: Accuracy: a checked, structured response

In [8]:
# ▶ CELL 7 · CHECKED + STRUCTURED: do not hand the user raw text
def check_answer(question, result):
    """Code, not hope: every number and every citation in the answer must come from the evidence."""
    evidence = json.dumps(result["evidence"], ensure_ascii=False)
    haystack = re.sub(r"(?<=\d),(?=\d)", "", evidence + " " + question)             # 12,00,000 -> 1200000
    answer, problems = result["answer"] or "", []
    for n in sorted(set(re.findall(r"\d[\d,]{3,}\d", answer))):                      # numbers with 4+ digits
        if n.replace(",", "") not in haystack:
            problems.append(f"number {n} is not in any tool result")
    for cite in sorted(set(re.findall(r"[A-Za-z0-9_]+ §\d+(?:\.\d+)*", answer))):          # citations such as Procurement_Policy §6.2
        if cite not in evidence:
            problems.append(f"citation {cite} was never retrieved")
    return problems

def structured(question, result=None):
    """Run the agent (or reuse a result) and return a CHECKED, STRUCTURED response: fields, not just text."""
    r = result or run_agent(question, verbose=False)
    problems = check_answer(question, r) if r["answer"] else ["no answer (step limit reached)"]
    return {"question": question, "answer": r["answer"], "tools_used": r["tools"], "steps": r["steps"],
            "checked": not problems, "problems": problems, "needs_human": bool(r["approval_tools"]) or bool(problems)}

reply = structured(CAPSTONE_Q, r)
print(json.dumps(reply, indent=2, ensure_ascii=False))

fake = {"answer": "The Director approved it for ₹15,00,000 [Procurement_Policy §9.9].", "evidence": r["evidence"]}
print("\nA made-up answer is caught →", check_answer(CAPSTONE_Q, fake))

{
  "question": "PO 4500012348 is blocked. Why, who must approve it, what does our policy say about price variance, and who is the vendor contact?",
  "answer": "PO 4500012348 is blocked due to price variance. The policy states that price variances above 1% block the invoice for review [Procurement_Policy §6.2]. This PO requires Director approval as it exceeds ₹10,00,000 [Procurement_Policy §3.3]. The vendor contact is Lakshmi Iyer at lakshmi@chennai.example.",
  "tools_used": [
    "get_order",
    "search_policy",
    "get_vendor"
  ],
  "steps": 4,
  "checked": true,
  "problems": [],
  "needs_human": false
}

A made-up answer is caught → ['number 15,00,000 is not in any tool result', 'citation Procurement_Policy §9.9 was never retrieved']


> ✅ **Code checks the answer**: every number and every citation must appear in the tools' results. A made-up figure or a clause nobody retrieved is flagged, and `needs_human` tells the app to ask a person.

### ▶ CELL 8: Cost and speed

In [9]:
# ▶ CELL 8 · COST AND SPEED: the three approaches compared
jobs = [("prompt only",   lambda: ask_ai(f"Summarise in one sentence:\n{EMAIL}", temperature=0)),
        ("RAG",           lambda: rag_answer("Who approves a PO above ₹10 lakh?")),
        ("agent + tools", lambda: run_agent(CAPSTONE_Q, verbose=False))]
print(f"{'approach':<15}{'model calls':<13}{'tokens in':<11}{'tokens out':<12}seconds")
for label, job in jobs:
    reset_usage()
    job()
    u = usage_total
    print(f"{label:<15}{u['calls']:<13}{u['input_tokens']:<11}{u['output_tokens']:<12}{u['seconds']}")

approach       model calls  tokens in  tokens out  seconds


prompt only    1            61         35          58.08


RAG            1            223        12          9.35


agent + tools  4            3446       284         77.69


> ⏱️ More steps means more model calls, more tokens and more waiting. With a paid cloud model, **tokens are money**. This is why we start simple and add an agent *only when the problem needs it*.

### ▶ CELL 9: Privacy and human oversight

In [10]:
# ▶ CELL 9 · PRIVACY AND OVERSIGHT: where does the data go, and who approves a change?
from urllib.parse import urlparse
host = urlparse(OLLAMA_URL).hostname
print(f"Model runs at {host} →", "🔒 this laptop: nothing leaves it" if host in ("localhost", "127.0.0.1") else "⚠️ another machine: check that it is approved")
print(f"RAG sends only 3 of {len(chunks)} chunks to the model, not the whole library.\n")

def ask_person(name, args):
    """Pause the agent until a human decides. (Set BOOTCAMP_AUTO_APPROVE=yes|no only for automated tests.)"""
    print(f"\n🛑 APPROVAL NEEDED: the agent wants to run  {name}({args})")
    answer = os.environ.get("BOOTCAMP_AUTO_APPROVE") or input("   Approve? Type yes or no: ")
    ok = answer.strip().lower() in ("y", "yes")
    print("   ✅ Approved by a person." if ok else "   ⛔ Declined. Nothing was changed.")
    return ok

def gate(tool, args):
    if tool["annotations"].get("readOnlyHint"):
        return True                                       # reading is safe: let it run
    return ask_person(tool["name"], args)                 # changing data: a person decides

r = run_agent("Create a ticket for PO 4500012348 asking the buyer to resolve the price variance.")
print("\nTools that needed approval:", r["approval_tools"])

Model runs at localhost → 🔒 this laptop: nothing leaves it
RAG sends only 3 of 39 chunks to the model, not the whole library.



🧠 THINK   (step 1): I need to create a ticket for the specified purchase order and note the reason.

🛑 APPROVAL NEEDED: the agent wants to run  create_ticket({'po_number': '4500012348', 'note': 'Ask the buyer to resolve the price variance'})
   ✅ Approved by a person.
🛠️  ACT     : create_ticket({'po_number': '4500012348', 'note': 'Ask the buyer to resolve the price variance'})
👀 OBSERVE: {"ticket_id": "TCK-0010", "po_number": "4500012348", "note": "Ask the buyer to resolve the price variance", "status": "Open", "created": "2026-10-29T23:10:07"}


🧠 THINK   (step 2): The ticket was successfully created for the purchase order.
✅ ANSWER : A ticket (TCK-0010) has been created for PO 4500012348 with the note: 'Ask the buyer to resolve the price variance'. The ticket is now open and was created on 2026-10-29.

Tools that needed approval: ['create_ticket']


> 🔒 **Privacy:** a local model keeps data on the laptop. Cloud tools (NotebookLM, Antigravity) need approval and **mock data only**. **Oversight:** reads run alone; any change waits for a person.

---
# 4️⃣ Capstone: your SAP Operations Assistant (40 min)
Pick **one SAP process**, ask it a multi-step question, test it, and fill in your pitch card. Change only the lines marked **← change me**.

### ▶ CELL 10: Pick a process and ask

In [11]:
# ▶ CELL 10 · YOUR ASSISTANT: pick one SAP process and ask it a multi-step question
USE_CASES = {
    "invoice exception":  "PO 4500012348 has a price variance. Why is it blocked, who must approve it, and what does our policy say?",
    "PO delay analysis":  "Which orders for plant 2000 are delayed, and who is the vendor contact for the first one?",
    "IDoc error triage":  "An IDoc is stuck at status 51. What does it mean, and who do we involve if the cause is the connection?",
    "vendor query":       "Who is the vendor contact for PO 4500012346, and what is the status of that order?",
    "month-end close":    "Which orders for plant 1000 are blocked, and what does our policy say about the 3-way match before payment?",
}
MY_USE_CASE = "invoice exception"                       # ← change me (one of the five above)
MY_QUESTION = USE_CASES[MY_USE_CASE]                    # ← or write your own question here

MY_RESULT = run_agent(MY_QUESTION)
reply = structured(MY_QUESTION, MY_RESULT)
print("\n📋 Checked:", reply["checked"], "| Needs a human:", reply["needs_human"], "| Problems:", reply["problems"])

🧠 THINK   (step 1): I need to check the status and block reason of PO 4500012348, determine the approver based on its value, and find the relevant policy clause.
🛠️  ACT     : get_order({'po_number': '4500012348'})
👀 OBSERVE: {"po_number": "4500012348", "vendor_id": "V300", "vendor_name": "Chennai Components", "plant": "1000", "material": "M-400 Gearbox", "qty": 10, "value_inr": 1200000, "delivery_date": "2026-09-30", "status": "Blocked", "block_reason": "Price variance"}


🧠 THINK   (step 2): I now know the block reason, value, and vendor. I need to find the approver based on the value and check the policy for price variance rules.
🛠️  ACT     : check_approval({'value_inr': 1200000})
👀 OBSERVE: {"value_inr": 1200000, "approver": "Director", "policy": "3.3"}


🧠 THINK   (step 3): I have the block reason, approver, and policy clause. I can now provide the final answer.
✅ ANSWER : PO 4500012348 is blocked due to a price variance. The approver is the Director, as per policy [Procurement_Policy §3.3]. The policy states that price variances require review and approval by the Director. The vendor is Chennai Components (V300).

📋 Checked: False | Needs a human: True | Problems: ['citation Procurement_Policy §3.3 was never retrieved']


> 🏁 Try the others. Notice which ones need only the **knowledge tool**, which need only **MCP tools**, and which need **both**.

### ▶ CELL 11: Test your assistant: 5 questions, scored in code

In [12]:
# ▶ CELL 11 · TEST PLAN: direct · reworded · not in the documents. The CODE checks tools and answer.
NOT_FOUND = ("not in the documents", "not found", "no information", "cannot", "do not know", "don't know", "not available", "no tool")
TESTS = [  # (type, question, tools that must be used, a word the answer must contain, or None for "must say it is missing")
    ("direct",   "What is the status of PO 4500012345?",                                  {"get_order"},                  "blocked"),
    ("direct",   "What does IDoc status 51 mean?",                                        {"search_policy"},              "not posted"),
    ("direct",   "Who is the vendor contact for PO 4500012346?",                          {"get_order", "get_vendor"},    "rahul"),
    ("reworded", "Which senior person has to sign off on PO 4500012348 because it is so large?", {"get_order", "check_approval"}, "director"),
    ("missing",  "What is the notice period for ending a vendor contract?",               set(),                          None),
]
points = 0
for kind, question, need, word in TESTS:
    r = run_agent(question, verbose=False)
    answer = (r["answer"] or "").lower()
    ok = need <= set(r["tools"]) and bool(answer) and ((word in answer) if word else any(p in answer for p in NOT_FOUND))
    points += ok
    print(f"{'✅' if ok else '❌'} {kind:<8} {question}")
    print(f"     tools : {r['tools']}   checked: {not check_answer(question, r) if r['answer'] else False}")
    print(f"     answer: {(r['answer'] or '—')[:140]}\n")
print(f"Score: {points} / {len(TESTS)}")

✅ direct   What is the status of PO 4500012345?
     tools : ['get_order']   checked: True
     answer: The status of PO 4500012345 is Blocked due to a Credit check.



✅ direct   What does IDoc status 51 mean?
     tools : ['search_policy']   checked: True
     answer: IDoc status 51 means the IDoc reached SAP but the application document was not posted. Typical causes include missing master data, a closed 



✅ direct   Who is the vendor contact for PO 4500012346?
     tools : ['get_order', 'get_vendor']   checked: True
     answer: The vendor contact for PO 4500012346 is Rahul Verma, and his email is rahul@bharat.example.



✅ reworded Which senior person has to sign off on PO 4500012348 because it is so large?
     tools : ['get_order', 'check_approval']   checked: False
     answer: The Director must sign off on PO 4500012348 as it exceeds the threshold for approval under [Procurement_Policy §3.3].



❌ missing  What is the notice period for ending a vendor contract?
     tools : ['search_policy']   checked: True
     answer: The notice period for ending a vendor contract is not specified in the provided policy sections. Please consult the full procurement policy 

Score: 4 / 5


> 📋 A ❌ is a good lesson. Was the **wrong tool** chosen? Was the **right clause not retrieved**? Or did the model **ignore** the result? The printed tools tell you where to look.

### ▶ CELL 12: Your 5-minute pitch card

In [13]:
# ▶ CELL 12 · YOUR 5-MINUTE PITCH: the card fills itself from what you built
TEAM = "Team 1"                                         # ← change me
GUARDRAILS_I_REQUIRE = [                                # ← change me: the guardrails you would insist on
    "Read-only tools by default",
    "A person approves any ticket or change",
    f"A step limit of {MAX_STEPS}",
    "Answers checked in code: numbers and citations must come from the tools",
    "Mock data only; a local model for anything sensitive",
]
print(f"""
╔══════════════════════ {TEAM} · SAP OPERATIONS ASSISTANT ══════════════════════
 1. THE PROCESS   : {MY_USE_CASE}
 2. THE PROMPT    : {ROLE[:110]}…
                    Format: {FORMAT}
 3. LIVE DEMO     : {MY_QUESTION}
                    tools used: {' → '.join(MY_RESULT['tools'])}   (test score: {points} / {len(TESTS)})""")
print(" 4. GUARDRAILS    :")
for g in GUARDRAILS_I_REQUIRE:
    print("      •", g)
print("╚" + "═" * 78)
mcp.close()


╔══════════════════════ Team 1 · SAP OPERATIONS ASSISTANT ══════════════════════
 1. THE PROCESS   : invoice exception
 2. THE PROMPT    : You are the SAP Operations Assistant for Nova Industries. You handle multi-step questions about purchase order…
                    Format: Write the final answer in at most four short sentences. Start with the main finding.
 3. LIVE DEMO     : PO 4500012348 has a price variance. Why is it blocked, who must approve it, and what does our policy say?
                    tools used: get_order → check_approval   (test score: 4 / 5)
 4. GUARDRAILS    :
      • Read-only tools by default
      • A person approves any ticket or change
      • A step limit of 6
      • Answers checked in code: numbers and citations must come from the tools
      • Mock data only; a local model for anything sensitive
╚══════════════════════════════════════════════════════════════════════════════


---
## ✅ Remember three things
1. **Start simple.** Prompt first, then add knowledge (RAG), then an agent, and only when the problem needs it.
2. **One assistant = parts you already know:** prompt and rules + agent + RAG + MCP tools + a checked answer.
3. **Business concerns decide success:** accuracy checks, cost, speed, data privacy and human oversight.

> 🌱 **Keep learning with the same free tools.** See `Class_8_Capstone/ROADMAP.md`.

---
## 🛠️ Trainer troubleshooting
| Message | Cause | Fix |
|---|---|---|
| `Cannot reach Ollama at http://localhost:11434` | Ollama isn't running | Windows: Start menu → **Ollama**. Mac: open the Ollama app. Or in a terminal: `ollama serve` |
| `Model '…' is not downloaded` | The model in `bootcamp_config.json` isn't pulled | `ollama pull <name>` (see `ollama list`) |
| `ModuleNotFoundError: bootcamp_llm` or `mcp_client` | The notebook is not running from its own folder | Open the **Day_4** folder as the workspace, then pick the **`.venv`** kernel |
| `The MCP server stopped before answering` | The server file crashed or the path is wrong | Run it by hand to see the error: `python Class_7_MCP_Guardrails/mcp_server/sap_orders_server.py` (press Ctrl+C to stop) |
| `Blocked by a guardrail: 'create_ticket' …` | The read-only gate did its job | Expected in CELL 8. CELL 9 swaps in human approval |
| `This server is READ-ONLY` | The server was started without `--allow-write` | Expected until CELL 9, which restarts it with writes enabled |
| `The model did not return valid JSON` | Small models sometimes break the format | Run the cell again. This is a real lesson: agents need validation |
| The agent picks the wrong tool | Normal with small models | Try a larger `model` in the config (7B or 8B follows tool menus better) |
| Antigravity / VS Code do not list the server | The config has no absolute path for this laptop | Run `python setup/make_mcp_config.py`, then restart the editor |